[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/grabuffo/brain-modeling-course/blob/main/notebooks/D1_C3_NB05_mpr_mean_field.ipynb)

# Day 1 · Chapter 3 · Notebook 05 — A population in two equations (Montbrió–Pazó–Roxin)

**Goal.** In NB03 we simulated 10 000 QIF neurons and saw collective oscillations, a high-activity
state, bistability and bursting. We ended with a question: for large $N$ the population rate is a smooth
curve, can we write an equation directly for it?

For QIF neurons with Lorentzian heterogeneity the answer is yes, and the equation is **exact**:
two ordinary differential equations, one for the firing rate $r$ and one for the voltage $v$
(Montbrió, Pazó & Roxin 2015, "MPR").

In this notebook we
1. see *why* two numbers are enough: the voltages of all neurons always follow a Lorentzian shape;
2. reproduce **Fig. 2 of MPR** (network vs mean field, step and sinusoidal input);
3. use the two equations to *understand* the network: fixed points, phase plane, bifurcation diagram,
   hysteresis, and the **phase diagram (MPR Fig. 1)** — analyses that are impossible with 10 000 equations.

In [ ]:
# --- Setup (run once) -------------------------------------------------------
# On Google Colab this downloads the course code; on your laptop it does nothing.
import sys, os
if "google.colab" in sys.modules and not os.path.exists("brain-modeling-course"):
    !git clone -q https://github.com/grabuffo/brain-modeling-course.git
    %cd brain-modeling-course/notebooks
    !pip -q install numba ipywidgets
sys.path.insert(0, os.path.abspath(".."))   # so that `import src` works

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact
from src import plotting
from src.plotting import slider
plotting.setup()
import time
from src import networks, meanfield

## 1. The firing-rate equations

Network (NB03):
$\;\dfrac{dv_j}{dt} = v_j^2 + \eta_j + J\,r(t) + I(t)$, $\;j=1\dots N$, with $\eta_j$ Lorentzian (centre $\bar\eta$, half-width $\Delta$).

Mean field, for $N\to\infty$:

$$
\boxed{\;
\frac{dr}{dt} = \frac{\Delta}{\pi} + 2\,r\,v, \qquad
\frac{dv}{dt} = v^2 + \bar\eta + J\,r + I(t) - \pi^2 r^2
\;}
$$

* $r(t)$: population firing rate. $v(t)$: the typical membrane voltage (centre of the voltage distribution).
* The $v$ equation looks like **one QIF neuron** with the average excitability $\bar\eta$, plus the
  recurrent input $Jr$, plus a new term $-\pi^2 r^2$: neurons that fire are reset to very negative
  voltages and pull the population voltage down.
* The $r$ equation: the rate grows when the typical voltage is positive ($v>0$, neurons are on their way to
  spike) and decays when it is negative. $\Delta/\pi$ is the contribution of the heterogeneity: some neurons
  always fire, so $r>0$.

**Where does it come from?** (derivation in MPR 2015, not needed here.) For QIF neurons the distribution
of voltages, if it starts Lorentzian, *stays* Lorentzian forever (this is the Ott–Antonsen ansatz). A Lorentzian
is fully described by two numbers, its centre and its half-width, and for the QIF network these are
exactly $v$ and $\pi r$:

$$
\rho(v_j, t) = \frac{1}{\pi}\,\frac{\pi r(t)}{\big(v_j - v(t)\big)^2 + \big(\pi r(t)\big)^2}.
$$

The width is $\pi r$ because the neurons that fire are the ones far out in the right tail of the
distribution: a wider distribution means more neurons crossing to $+\infty$ per unit time.

## 2. Seeing the Lorentzian in the network

We run the network of NB03 ($N=10^4$, step $I=3$ from $t=0$) and take snapshots of the voltages of
**all** neurons. On top we draw the Lorentzian predicted by the mean field at that time
(centre $v(t)$ and half-width $\pi r(t)$ from the two equations).

In [ ]:
dt = 1e-4
T0, T = -10.0, 70.0
t = np.arange(T0, T, dt)
I_step = np.where((t >= 0) & (t < 30), 3.0, 0.0)
I_sin = 3.0 * np.sin(np.pi * t / 20.0) * (t >= 0)

r_low, v_low = meanfield.mpr_fixed_points(eta_bar=-5.0, J=15.0)[0]      # low-activity state
t_snap = np.array([-9.99, -5.0, 0.5, 2.0, 2.6, 20.0])
tb, r_net, v_net, st, si, snaps = networks.qif_network(I_step, N=10000, dt=dt, bin_size=0.02,
                                                      snap_times=t_snap - T0, seed=0)
r_mf, v_mf = meanfield.mpr(I_step, dt=dt, r0=r_low, v0=v_low)

fig, ax = plt.subplots(2, 3, figsize=(12, 6))
for a_, ts, x in zip(ax.flat, t_snap, snaps):
    x = x[~np.isnan(x)]
    i = int(round((ts - T0) / dt))
    w = np.pi * r_mf[i]
    lo, hi = v_mf[i] - 6 * w - 1.0, v_mf[i] + 6 * w + 1.0
    a_.hist(x, bins=np.linspace(lo, hi, 70), density=True, color="C0", alpha=0.6, label="network")
    vv = np.linspace(lo, hi, 500)
    a_.plot(vv, meanfield.lorentzian_v_density(vv, v_mf[i], r_mf[i]), "C1", lw=2, label="mean field")
    a_.set(title=f"t = {ts:g}   (r = {r_mf[i]:.2f}, v = {v_mf[i]:.2f})", xlabel="voltage $v_j$")
ax[0, 0].legend(fontsize=8); ax[0, 0].set_ylabel("density"); ax[1, 0].set_ylabel("density")
plt.tight_layout()

* $t=-9.99$: the network was started with voltages uniform in $[-2,-1]$, **not** a Lorentzian.
* $t=-5$: a few time units later the network has *forgotten* its initial condition and its voltages are
  Lorentzian, exactly as predicted. The Lorentzian shape is an **attractor** of the network dynamics, so
  the two equations hold after a short transient whatever the initial condition.
* $t=0.5$ to $2.6$: during the switch to high activity the distribution moves right and widens (more
  firing = wider), and it remains Lorentzian.
* $t=20$: the high-activity state, a wide Lorentzian centred near 0.

A useful check that you can do with any data: for a Lorentzian, the **median** is the centre and **half the
interquartile range** is the half-width. Compare them with $v$ and $\pi r$:

In [ ]:
print("    t   |  median v_j   MF v   |  half IQR   MF pi*r")
for ts, x in zip(t_snap[1:], snaps[1:]):
    x = x[~np.isnan(x)]; i = int(round((ts - T0) / dt))
    q1, q2, q3 = np.percentile(x, [25, 50, 75])
    print(f"{ts:6.1f} | {q2:8.2f}  {v_mf[i]:8.2f}   | {(q3 - q1) / 2:7.2f}  {np.pi * r_mf[i]:8.2f}")

(At $t=2.6$, the peak of the first population spike, the network lags the mean field a little: the rate
changes very fast there, and a tiny shift in time gives a large difference.)

## 3. Reproducing MPR Fig. 2: 10 000 neurons vs 2 equations

In [ ]:
fig, ax = plt.subplots(4, 2, figsize=(12, 8.5), sharex=True, gridspec_kw=dict(height_ratios=[1.6, 1.3, 1.3, 0.5]))
timing = {}
for c, (name, I_ext) in enumerate([("step", I_step), ("sine", I_sin)]):
    t0 = time.perf_counter()
    tb, r_net, v_net, st, si = networks.qif_network(I_ext, N=10000, dt=dt, bin_size=0.02, n_rec=300, seed=0)
    timing["network"] = time.perf_counter() - t0
    t0 = time.perf_counter()
    r_mf, v_mf = meanfield.mpr(I_ext, dt=dt, r0=r_low, v0=v_low)
    timing["mean field"] = time.perf_counter() - t0
    ax[0, c].plot(st + T0, si, ".k", ms=0.6)
    ax[1, c].plot(t, r_mf, "C1", lw=2.5, alpha=0.8, label="mean field (2 ODEs)")
    ax[1, c].plot(tb + T0, r_net, "k", lw=0.6, label="network, N = 10 000")
    ax[2, c].plot(t, v_mf, "C1", lw=2.5, alpha=0.8)
    ax[2, c].plot(tb + T0, v_net, "k", lw=0.6)
    ax[3, c].plot(t, I_ext, "C3")
ax[0, 0].set(ylabel="neuron #", title="step input"); ax[0, 1].set(title="sinusoidal input")
ax[1, 0].set_ylabel("rate r"); ax[1, 0].legend(fontsize=8, loc="upper right")
ax[2, 0].set_ylabel("voltage v"); ax[2, 0].set_ylim(-3, 2); ax[2, 1].set_ylim(-3, 2)
ax[3, 0].set_ylabel("I(t)")
for a_ in ax[3]: a_.set_xlabel("time ($\\tau_m$)")
plt.tight_layout()
print(f"time for one run:  network {timing['network']:.2f} s,  mean field {timing['mean field']:.3f} s")

The two equations reproduce everything we saw in the network: the onset of high activity, the
**damped collective oscillations**, the **memory** after the step, the **bursts** under sinusoidal input.
The only things they miss are the small finite-size fluctuations of the network (the mean field is
$N=\infty$).

The mean field is thousands of times cheaper, but cost is not the main point. The main point is that
with two equations we can *analyse* the population with the tools of dynamical systems: fixed points,
stability, bifurcations. This is what the rest of the notebook does.

> **Why not a classic rate model?** Heuristic rate models (Wilson–Cowan, 1972) describe a population with
> one variable, the rate, and a sigmoid. They capture bistability but **not** the fast collective
> oscillations at the step onset, because those come from the interplay of rate *and* voltage (MPR 2015,
> Fig. 2 vs their rate-model comparison). The exact mean field keeps both.

## 4. Phase plane: why the network has two stable states

Set $I=0$. The **nullclines** are the curves where one variable does not change:

* $dr/dt=0$: $\;v = -\dfrac{\Delta}{2\pi r}$ (a hyperbola, always at negative $v$);
* $dv/dt=0$: $\;v = \pm\sqrt{\pi^2 r^2 - J r - \bar\eta - I}$ (two branches).

Fixed points are where the nullclines cross. Their stability comes from the eigenvalues $\lambda$ of the
Jacobian (as in NB02): negative real parts = stable; complex $\lambda$ = the trajectory spirals in
(**damped oscillation** with period $2\pi/|\mathrm{Im}\,\lambda|$).

In [ ]:
def nullclines(I, eta_bar=-5.0, J=15.0, delta=1.0, r=np.linspace(1e-3, 2.5, 3000)):
    vr = -delta / (2 * np.pi * r)
    arg = np.pi**2 * r**2 - J * r - eta_bar - I
    vp = np.where(arg >= 0, np.sqrt(np.clip(arg, 0, None)), np.nan)
    return r, vr, vp, -vp

def classify(r, v, J):
    lam = np.linalg.eigvals(meanfield.mpr_jacobian(r, v, J))
    stable = np.all(lam.real < 0)
    kind = ("stable " if stable else "unstable ") + ("focus" if np.any(lam.imag != 0) else ("node" if stable or np.all(lam.real > 0) else "saddle"))
    return lam, stable, kind

def phase_plane(ax, I, eta_bar=-5.0, J=15.0, delta=1.0, n_traj=True, vlim=(-3, 1.5), rmax=2.0):
    r, vr, vp, vm = nullclines(I, eta_bar, J, delta, np.linspace(1e-3, rmax, 3000))
    R, V = np.meshgrid(np.linspace(1e-3, rmax, 30), np.linspace(*vlim, 30))
    dR = delta / np.pi + 2 * R * V
    dV = V**2 + eta_bar + J * R + I - np.pi**2 * R**2
    ax.streamplot(R, V, dR, dV, color="0.8", density=1.0, linewidth=0.6, arrowsize=0.7)
    ax.plot(r, vr, "C0", lw=2, label="dr/dt = 0")
    ax.plot(r, vp, "C2", lw=2, label="dv/dt = 0"); ax.plot(r, vm, "C2", lw=2)
    if n_traj:   # trajectories from a few initial conditions, coloured by where they end up
        for r0 in (0.3, 1.6):
            for v0 in np.linspace(vlim[0] + 0.3, vlim[1] - 0.3, 6):
                rr, vv = meanfield.mpr(np.full(20000, I), dt=1e-3, r0=r0, v0=v0, delta=delta, eta_bar=eta_bar, J=J)
                ax.plot(rr, vv, color="C3" if rr[-1] > 0.5 else "C4", lw=0.7, alpha=0.7)
    for rs, vs in meanfield.mpr_fixed_points(eta_bar, J, delta, I):
        lam, stable, kind = classify(rs, vs, J)
        ax.plot(rs, vs, "o", ms=10, mfc="k" if stable else "w", mec="k", zorder=5)
        ax.annotate(kind, (rs, vs), textcoords="offset points", xytext=(8, 8), fontsize=8)
    ax.set(xlim=(0, rmax), ylim=vlim, xlabel="rate r", ylabel="voltage v")

fig, ax = plt.subplots(figsize=(7, 5))
phase_plane(ax, I=0.0)
ax.set_title(r"MPR phase plane, $\bar\eta=-5$, $J=15$, $\Delta=1$, $I=0$")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()

for rs, vs in meanfield.mpr_fixed_points(-5.0, 15.0):
    lam, stable, kind = classify(rs, vs, 15.0)
    extra = f",  damped-oscillation period 2pi/|Im lambda| = {2 * np.pi / abs(lam[0].imag):.2f}" if np.iscomplex(lam[0]) else ""
    print(f"r* = {rs:.3f}, v* = {vs:.3f}: {kind:16s} lambda = {np.round(lam, 2)}{extra}")

Three fixed points:

* **low-activity state** (stable node): few neurons fire;
* **saddle** (unstable): the *threshold of the population*. Its stable manifold separates the initial
  conditions that go to the low state (purple) from those that go to the high state (red), as the threshold
  of the single QIF neuron in NB02 separated rest from spiking;
* **high-activity state** (stable **focus**): trajectories spiral into it, and this spiralling is the
  **damped collective oscillation** we saw in the network.

The eigenvalues are a *prediction*: the period of the collective oscillation. Let us check it against the
network during the step ($I=3$):

In [ ]:
rs, vs = meanfield.mpr_fixed_points(-5.0, 15.0, I=3.0)[-1]
lam, *_ = classify(rs, vs, 15.0)
print(f"predicted period of the collective oscillation at I = 3: {2 * np.pi / abs(lam[0].imag):.3f}")

tb, r_net, *_ = networks.qif_network(I_step, N=10000, dt=dt, bin_size=0.02, seed=0)
tt = tb + T0
m = (tt > 5) & (tt < 25)                       # during the step, after the first population spikes
x = r_net[m] - r_net[m].mean()
freqs = np.fft.rfftfreq(x.size, d=tt[1] - tt[0])
f_peak = freqs[np.argmax(np.abs(np.fft.rfft(x)))]
print(f"measured in the network (dominant frequency of r(t), t in 5-25): period {1 / f_peak:.3f}")

## 5. Explore: the phase plane as you change the input

Left: phase plane. Right: the **bifurcation diagram**, the fixed-point rate $r^*$ as a function of the input
$I$ (black = stable, red = unstable), with the current $I$ marked.

* Increase $I$ slowly from 0. At which value do the low state and the saddle **collide and disappear**?
  This is a **saddle-node bifurcation** of the population, the same bifurcation as the single QIF neuron in NB02,
  one level up.
* Decrease $I$ below 0. Which state disappears now, and where?
* Set $J=5$. Is the population still bistable for some $I$?
* Increase $\Delta$ (more heterogeneous neurons). What happens to the bistable range?

In [ ]:
def bif_diagram(ax, eta_bar, J, delta, I_range=(-4, 6), n=600):
    br = meanfield.mpr_branches(np.linspace(*I_range, n), eta_bar, J, delta)
    for stable, c in [(True, "k"), (False, "C3")]:
        b = np.array([(x[0], x[1]) for x in br if x[3] == stable] or [(np.nan, np.nan)])
        ax.plot(b[:, 0], b[:, 1], ".", c=c, ms=2, label="stable" if stable else "unstable")
    ax.set(xlabel="input I", ylabel="fixed-point rate r*", xlim=I_range, ylim=(0, 2))

@interact(I=slider(0.0, -3.0, 5.0, 0.1, "input I"),
          J=slider(15.0, 0.0, 25.0, 0.5, "coupling J"),
          eta_bar=slider(-5.0, -10.0, 2.0, 0.25, "mean excitability eta_bar"),
          delta=slider(1.0, 0.2, 3.0, 0.1, "heterogeneity Delta"))
def explore_phase_plane(I, J, eta_bar, delta):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
    phase_plane(ax[0], I, eta_bar, J, delta)
    ax[0].set_title(f"phase plane, I = {I}")
    bif_diagram(ax[1], eta_bar, J, delta)
    ax[1].axvline(I, c="C0", lw=1)
    for rs, vs in meanfield.mpr_fixed_points(eta_bar, J, delta, I):
        ax[1].plot(I, rs, "o", mfc="w", mec="C0", ms=9)
    ax[1].legend(fontsize=8, loc="upper left"); ax[1].set_title("bifurcation diagram")
    plt.tight_layout(); plt.show()

## 6. Hysteresis: the network follows the bifurcation diagram

Now we ramp the input slowly up from $I=-4$ to $I=4$ and back down, in both the network ($N=5000$) and the
mean field, and plot the rate against the input.

In [ ]:
dt2 = 5e-4; T_ramp = 150.0
t2 = np.arange(0, 2 * T_ramp, dt2)
I_ramp = np.where(t2 < T_ramp, -4 + 8 * t2 / T_ramp, 4 - 8 * (t2 - T_ramp) / T_ramp)
tb2, r_net2, *_ = networks.qif_network(I_ramp, N=5000, dt=dt2, bin_size=0.1, seed=0)
r_mf2, _ = meanfield.mpr(I_ramp, dt=dt2, r0=0.05, v0=-3.0)

br = meanfield.mpr_branches(np.linspace(-4, 4, 801), -5.0, 15.0)
I_bi = np.array([x[0] for x in br if not x[3]])          # inputs with an unstable (saddle) branch
I_down, I_up = I_bi.min(), I_bi.max()

fig, ax = plt.subplots(figsize=(8, 5))
bif_diagram(ax, -5.0, 15.0, 1.0, I_range=(-4.2, 4.2))
up = tb2 < T_ramp
I_at_bin = np.interp(tb2, t2, I_ramp)
ax.plot(I_at_bin[up], r_net2[up], "C0", lw=0.8, alpha=0.8, label="network, ramp up")
ax.plot(I_at_bin[~up], r_net2[~up], "C2", lw=0.8, alpha=0.8, label="network, ramp down")
ax.plot(I_ramp, r_mf2, "C1", lw=1.2, label="mean field")
for x, lab in [(I_down, "saddle-node"), (I_up, "saddle-node")]:
    ax.axvline(x, c="0.5", ls=":"); ax.text(x + 0.05, 1.85, f"{lab}\nI = {x:.2f}", fontsize=8)
ax.annotate("", xy=(I_up + 0.25, 1.2), xytext=(I_up + 0.25, 0.3), arrowprops=dict(arrowstyle="->", color="C0"))
ax.annotate("", xy=(I_down - 0.25, 0.2), xytext=(I_down - 0.25, 0.75), arrowprops=dict(arrowstyle="->", color="C2"))
ax.legend(fontsize=8, loc="center left"); ax.set_title("hysteresis loop")
plt.tight_layout()
print(f"bistable for {I_down:.2f} < I < {I_up:.2f}")

Going up, the network stays on the low branch until the low state disappears (saddle-node at $I\approx1.9$),
then jumps to the high branch, ringing as it spirals into the focus. Going down, it stays high until the high
state disappears (saddle-node at $I\approx-0.7$). Between the two the state depends on the history: the
population has **memory**.

This explains the bursts of the sinusoidal input in Fig. 2: the input sweeps back and forth across the
bistable range, and the network jumps up and down at the two saddle-nodes.

## 7. The phase diagram (MPR Fig. 1)

Every pair $(\bar\eta, J)$ is a different network. The phase diagram shows, for each pair, how many stable
states the network has. The boundary of the bistable region is made of saddle-node bifurcations, and it can be
written in closed form (MPR 2015, Eq. 14), parametrised by the rate $r$:

$$
\bar\eta = -\pi^2 r^2 - \frac{3\Delta^2}{4\pi^2 r^2}, \qquad J = 2\pi^2 r + \frac{\Delta^2}{2\pi^2 r^3}.
$$

Only the ratios $\bar\eta/\Delta$ and $J/\sqrt\Delta$ matter (change $\Delta$ and the whole diagram rescales),
so we plot in these units.

In [ ]:
eta_grid = np.linspace(-15, 5, 161); J_grid = np.linspace(0, 30, 121)
n_fp = np.zeros((J_grid.size, eta_grid.size)); focus = np.zeros_like(n_fp)
for i, Jg in enumerate(J_grid):
    for k, eg in enumerate(eta_grid):
        fps = meanfield.mpr_fixed_points(eg, Jg)
        n_fp[i, k] = len(fps)
        focus[i, k] = any(classify(rs, vs, Jg)[2] == "stable focus" for rs, vs in fps)

def phase_diagram(ax):
    ax.contourf(eta_grid, J_grid, focus, levels=[0.5, 1.5], colors=["C0"], alpha=0.12)
    ax.contourf(eta_grid, J_grid, n_fp, levels=[2.5, 3.5], colors=["C1"], alpha=0.35)
    e_sn, J_sn = meanfield.mpr_saddle_node_curve(1.0)
    ax.plot(e_sn, J_sn, "k", lw=1.5)
    ax.text(-13, 25, "bistable", fontsize=10); ax.text(-13, 8, "low activity\n(one stable node)", fontsize=9)
    ax.text(0.5, 20, "high activity\n(one stable state)", fontsize=9, ha="left")
    ax.set(xlim=(-15, 5), ylim=(0, 30), xlabel=r"$\bar\eta/\Delta$", ylabel=r"$J/\sqrt{\Delta}$")

fig, ax = plt.subplots(figsize=(7, 5))
phase_diagram(ax)
ax.plot(-5, 15, "*", ms=14, c="C3", mec="k", label=r"NB03 network ($\bar\eta=-5$, $J=15$)")
ax.annotate("", xy=(-2, 15), xytext=(-5, 15), arrowprops=dict(arrowstyle="->", color="C3", lw=1.5))
ax.text(-4.6, 16, "during the step\nI = 3", fontsize=8, color="C3")
e_sn, J_sn = meanfield.mpr_saddle_node_curve(1.0); i = np.argmin(J_sn)
ax.plot(e_sn[i], J_sn[i], "ko", ms=5); ax.annotate("cusp", (e_sn[i], J_sn[i]), xytext=(6, -12), textcoords="offset points", fontsize=8)
ax.legend(fontsize=8, loc="lower left"); ax.set_title("MPR phase diagram (Fig. 1)\nblue shading: a stable focus exists (damped collective oscillations)", fontsize=10)
plt.tight_layout()
print(f"cusp at eta_bar/Delta = {e_sn[i]:.3f} (= -sqrt(3)), J/sqrt(Delta) = {J_sn[i]:.3f}")

* An input $I$ enters the equations exactly like $\bar\eta$ (look at the $v$ equation): **a step of input
  moves the network horizontally** in the phase diagram. The step $I=3$ moves our network from the bistable
  region into the high-activity region, and removing it brings it back into the bistable region, on the high
  branch.
* The two saddle-node curves meet at a **cusp**. Below the cusp ($J/\sqrt\Delta < 7.8$) there is no
  bistability: the recurrent excitation is too weak compared to the heterogeneity.
* Most of the plane where the network is active has a stable **focus**: collective oscillations are damped,
  never sustained. Sustained oscillations need more ingredients (synaptic delays or kinetics, inhibition;
  see the references). We will meet a model with sustained collective oscillations in NB06.

### Explore: pick a point in the phase diagram, compare network and mean field

The network has $N=5000$ neurons and a coarser time step (< 1 s per run). The step $I_0$ is applied for
$0<t<30$.

* Put the network below the cusp (e.g. $J=5$). Is there still memory after the step?
* Find a step $I_0$ that is too small to switch the network up. Where is the arrow in the phase diagram?
* Increase $\Delta$: the point moves in the rescaled diagram. Watch the oscillations and the memory.
* Where does the mean field disagree most with the network? (Try small $\Delta$.)

In [ ]:
@interact(J=slider(15.0, 0.0, 25.0, 0.5, "coupling J"),
          eta_bar=slider(-5.0, -10.0, 2.0, 0.25, "mean excitability eta_bar"),
          delta=slider(1.0, 0.2, 3.0, 0.1, "heterogeneity Delta"),
          I0=slider(3.0, -3.0, 6.0, 0.25, "step amplitude I0"))
def explore_network_vs_mf(J, eta_bar, delta, I0):
    dtx = 5e-4
    tx = np.arange(T0, T, dtx)
    Ix = np.where((tx >= 0) & (tx < 30), I0, 0.0)
    tb, r_n, v_n, *_ = networks.qif_network(Ix, N=5000, J=J, eta_bar=eta_bar, delta=delta, dt=dtx, bin_size=0.05, seed=0)
    fps = meanfield.mpr_fixed_points(eta_bar, J, delta)
    r_m, v_m = meanfield.mpr(Ix, dt=dtx, r0=fps[0, 0], v0=fps[0, 1], delta=delta, eta_bar=eta_bar, J=J)
    fig = plt.figure(figsize=(13, 4.6))
    a0 = fig.add_axes([0.05, 0.13, 0.27, 0.78]); a1 = fig.add_axes([0.40, 0.55, 0.57, 0.38]); a2 = fig.add_axes([0.40, 0.13, 0.57, 0.38], sharex=a1)
    phase_diagram(a0)
    x0, y0 = eta_bar / delta, J / np.sqrt(delta)
    a0.plot(x0, y0, "*", ms=14, c="C3", mec="k")
    a0.annotate("", xy=((eta_bar + I0) / delta, y0), xytext=(x0, y0), arrowprops=dict(arrowstyle="->", color="C3", lw=1.5))
    a1.plot(tb + T0, r_n, "k", lw=0.8, label="network N = 5000"); a1.plot(tx, r_m, "C1", lw=1.4, label="mean field")
    a1.set(ylabel="rate r", title=f"J={J}, eta_bar={eta_bar}, Delta={delta}, I0={I0}"); a1.legend(fontsize=8, loc="upper right")
    a2.plot(tb + T0, v_n, "k", lw=0.8); a2.plot(tx, v_m, "C1", lw=1.4)
    a2.set(ylabel="voltage v", xlabel="time ($\\tau_m$)", ylim=(min(-3, np.nanmin(v_m) - 0.5), 2))
    plt.setp(a1.get_xticklabels(), visible=False)
    plt.show()

## 8. When is the mean field exact, and when not?

The MPR equations are exact under three conditions. Each one is a modelling choice, and it is worth knowing
what happens when it is broken.

| Assumption | What breaks it | What happens |
|---|---|---|
| $N\to\infty$ | real populations are finite | the network fluctuates around the mean field (below) |
| Lorentzian heterogeneity | Gaussian or other $\eta$ distributions | the equations become an approximation (Homework 3) |
| all-to-all, instantaneous synapses | sparse connectivity, synaptic kinetics, delays | extensions exist (Devalle et al. 2017; Coombes & Byrne 2019) |

In [ ]:
dt2 = 5e-4; t2 = np.arange(T0, T, dt2); I2 = np.where((t2 >= 0) & (t2 < 30), 3.0, 0.0)
r_m, _ = meanfield.mpr(I2, dt=dt2, r0=r_low, v0=v_low)
fig, ax = plt.subplots(3, 1, figsize=(9, 5.5), sharex=True, sharey=True)
for a_, N in zip(ax, [200, 2000, 20000]):
    tb, r_n, *_ = networks.qif_network(I2, N=N, dt=dt2, bin_size=0.05, seed=1)
    a_.plot(tb + T0, r_n, "k", lw=0.7, label=f"network N = {N}"); a_.plot(t2, r_m, "C1", lw=1.4, label="mean field")
    a_.set(ylabel="rate r", ylim=(0, 3.5)); a_.legend(fontsize=8, loc="upper right")
ax[-1].set_xlabel("time ($\\tau_m$)")
plt.tight_layout()

The mean field is the limit the network approaches as $N$ grows. The fluctuations of finite networks are not
just noise to be ignored: near a bifurcation they can switch the network between states on their own
(Homework 4). We will come back to this in Chapter 4, because this sensitivity is what makes whole-brain models
near a bifurcation look realistic.

## 9. Take-home

* For QIF neurons with Lorentzian heterogeneity, a network of $N\to\infty$ neurons is described **exactly**
  by two equations, for the rate $r$ and the voltage $v$. The reason: the voltages always have a Lorentzian
  shape, with centre $v$ and width $\pi r$.
* The mean field reproduces the network (MPR Fig. 2) at a tiny fraction of the cost.
* More importantly, it makes the population **analysable**: the high state is a stable focus (collective
  oscillations, period predicted by the eigenvalues); bistability comes from two **saddle-node bifurcations**,
  which give hysteresis and bursting; the phase diagram (MPR Fig. 1) shows all the possible networks at a glance.
* The same saddle-node bifurcation organises the single neuron (NB02) and the population (here). Bifurcations
  are the common language across scales.

### Homework
*To do at home, after the class. In class we use the **Explore** sliders; these exercises take the same
questions further and usually need a few lines of your own code.*
1. **The rate is an average of single-neuron rates.** At a fixed point, each neuron receives the constant input
   $\eta_j + J r^*$ and fires at $\sqrt{\eta_j + J r^*}/\pi$ (NB02). Check numerically that the average of this
   over the Lorentzian distribution equals $r^*$ for the three fixed points at $\bar\eta=-5$, $J=15$
   (hint: `scipy.integrate.quad`).
2. **Inhibition.** Set $J=-10$. How many fixed points are there? Is the focus still there? Compare network and
   mean field for a step.
3. **Gaussian neurons.** Run the network with Gaussian excitabilities,
   `networks.qif_network(I, eta=np.random.default_rng(0).normal(-5, s, 10000))`, with the same half-width as the
   Lorentzian ($s = \Delta/\sqrt{2\ln 2}$). Compare with the MPR equations. What changes?
4. **Noise-driven switching.** With $N=200$, $\bar\eta=-5$, $J=15$, choose a constant input inside the bistable
   range but close to the upper saddle-node (e.g. $I=1.7$). Run for a long time. Does the small network switch
   to the high state without any step? Repeat with $N=5000$.
5. **Pulse threshold.** Starting from the low state at $I=0$, apply short pulses ($\Delta t = 1$) of increasing
   amplitude. Find the smallest pulse that switches the population to high activity, in the mean field and in the
   network. Relate it to the saddle in the phase plane.

### References
* Montbrió E, Pazó D, Roxin A (2015). Macroscopic description for networks of spiking neurons. *Phys Rev X* 5:021028.
* Ott E, Antonsen TM (2008). Low dimensional behavior of large systems of globally coupled oscillators. *Chaos* 18:037113.
* Luke TB, Barreto E, So P (2013). Complete classification of the macroscopic behavior of a heterogeneous network of theta neurons. *Neural Comput* 25:3207–3234.
* Devalle F, Roxin A, Montbrió E (2017). Firing rate equations require a spike synchrony mechanism to correctly describe fast oscillations in inhibitory networks. *PLoS Comput Biol* 13:e1005881.
* Coombes S, Byrne Á (2019). Next generation neural mass models. In *Nonlinear Dynamics in Computational Neuroscience*, Springer.
* Wilson HR, Cowan JD (1972). Excitatory and inhibitory interactions in localized populations of model neurons. *Biophys J* 12:1–24.